In [ ]:
import pandas as pd

df = pd.read_csv("/home/dell/ML-Learning/datasets/online_retail_II.csv", encoding="latin1")
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("Raw shape:", df.shape)
df.head()

Raw shape: (1067371, 8)


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   invoice      1067371 non-null  str    
 1   stockcode    1067371 non-null  str    
 2   description  1062989 non-null  str    
 3   quantity     1067371 non-null  int64  
 4   invoicedate  1067371 non-null  str    
 5   price        1067371 non-null  float64
 6   customer_id  824364 non-null   float64
 7   country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


In [4]:
df.isnull().sum()

invoice             0
stockcode           0
description      4382
quantity            0
invoicedate         0
price               0
customer_id    243007
country             0
dtype: int64

In [6]:
def fix_types(df):
    df["invoicedate"] = pd.to_datetime(df["invoicedate"], errors="coerce")
    df["description"] = df["description"].str.strip()
    df["invoice"] = df["invoice"].astype(str).str.strip()
    df["stockcode"] = df["stockcode"].astype(str).str.strip()
    return df

df = fix_types(df)

In [7]:
dupes = df[df.duplicated(keep=False)].sort_values(["invoice", "stockcode"])
dupes.head(20)

,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom


In [8]:
def remove_duplicates(df):
    before = len(df)
    df = df.drop_duplicates()
    print(f"Removed duplicates: {before - len(df)}")
    return df

df = remove_duplicates(df)

Removed duplicates: 34335


In [9]:
# invoice numbers starting with C are cancellations/returns
# these are real transactions, so we flag them instead of deleting
def flag_cancellations(df):
    df["is_cancelled"] = df["invoice"].str.startswith("C")
    print(f"Cancelled orders flagged: {df['is_cancelled'].sum()}")
    return df

df = flag_cancellations(df)

Cancelled orders flagged: 19104


In [11]:
def remove_non_products(df):
    junk_codes = ["POST", "DOT", "M", "C2", "BANK CHARGES",
                  "TEST001", "TEST002", "AMAZONFEE"]
    before = len(df)
    df = df[~df["stockcode"].str.upper().isin(junk_codes)]
    print(f"Removed non-product rows: {before - len(df)}")
    return df

df = remove_non_products(df)

Removed non-product rows: 0


In [38]:
# missing description -> row is unusable, drop it
# missing customer_id -> guest checkout, still valid, just flag it
def handle_missing_values(df):
    before = len(df)
    df = df.dropna(subset=["description"])
    print(f"Dropped rows with missing description: {before - len(df)}")

    df["has_customer_id"] = df["customer_id"].notna()
    print(f"Rows without customer id (kept, flagged): {(~df['has_customer_id']).sum()}")
    return df

df = handle_missing_values(df)

Dropped rows with missing description: 4262
Rows without customer id (kept, flagged): 228947


In [39]:
def remove_invalid_values(df):
    before = len(df)
    df = df[df["quantity"] != 0]
    df = df[df["price"] > 0]
    df = df[~((df["quantity"] < 0) & (~df["is_cancelled"]))]
    print(f"Removed invalid qty/price rows: {before - len(df)}")
    return df

df = remove_invalid_values(df)

Removed invalid qty/price rows: 1730


In [40]:
# IQR method - some rows have quantity in the tens of thousands
# which would badly skew averages/stats
def remove_outliers(df, columns):
    before = len(df)
    for col in columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        low = q1 - 3 * iqr
        high = q3 + 3 * iqr
        df = df[(df[col] >= low) & (df[col] <= high)]
    print(f"Removed outlier rows: {before - len(df)}")
    return df

df = remove_outliers(df, ["quantity", "price"])

Removed outlier rows: 62056


In [41]:
print(df["quantity"].describe())
print(df["price"].describe())

count    959655.000000
mean          6.189233
std           7.173978
min         -26.000000
25%           1.000000
50%           3.000000
75%          10.000000
max          37.000000
Name: quantity, dtype: float64
count    959655.000000
mean          3.058118
std           2.621279
min           0.001000
25%           1.250000
50%           2.100000
75%           4.130000
max          13.030000
Name: price, dtype: float64


In [42]:
# only cut truly implausible values, not just "large" ones
# based on manually inspecting the extremes earlier:
# - orders up to ~80,000 units were real wholesale purchases (paired with cancellations)
# - price outliers were mostly non-product fee codes, already removed earlier
def remove_extreme_outliers(df):
    before = len(df)
    df = df[df["quantity"].abs() <= 5000]   # generous cutoff, keeps real bulk orders
    df = df[df["price"] <= 1000]            # generous cutoff on price too
    print(f"Removed extreme outlier rows: {before - len(df)}")
    return df

df = remove_extreme_outliers(df)
print(df["quantity"].describe())
print(df["price"].describe())

Removed extreme outlier rows: 0
count    959655.000000
mean          6.189233
std           7.173978
min         -26.000000
25%           1.000000
50%           3.000000
75%          10.000000
max          37.000000
Name: quantity, dtype: float64
count    959655.000000
mean          3.058118
std           2.621279
min           0.001000
25%           1.250000
50%           2.100000
75%           4.130000
max          13.030000
Name: price, dtype: float64


In [ ]:
import pandas as pd

# ============================================================
# Online Retail II - Data Cleaning
# ============================================================
# Dataset: UK-based online retailer, transactions from
# Dec 2009 to Dec 2011. Source: UCI Machine Learning Repository.



# 1. Load data and fix column names

def load_data(path):
    df = pd.read_csv(path, encoding="latin1")
    # original column names have spaces and inconsistent casing
    # e.g. "Customer ID" -> "customer_id"
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    return df



# 2. Fix data types

def fix_types(df):
    # invoicedate comes in as plain text, need it as an actual
    # datetime so we can extract month/day/hour etc later
    df["invoicedate"] = pd.to_datetime(df["invoicedate"], errors="coerce")

    # strip extra whitespace from text columns
    df["description"] = df["description"].str.strip()
    df["invoice"] = df["invoice"].astype(str).str.strip()
    df["stockcode"] = df["stockcode"].astype(str).str.strip()
    return df



# 3. Remove duplicate rows

def remove_duplicates(df):
    before = len(df)
    df = df.drop_duplicates()
    print(f"Removed duplicates: {before - len(df)}")
    return df



# 4. Flag cancelled orders

# Invoice numbers starting with "C" are cancellations/returns.
# These are real business events, not bad data, so they are
# flagged instead of deleted. This lets later steps (like revenue
# calculations) filter them out only when needed.
def flag_cancellations(df):
    df["is_cancelled"] = df["invoice"].str.startswith("C")
    print(f"Cancelled orders flagged: {df['is_cancelled'].sum()}")
    return df



# 5. Remove non-product rows

# Some stockcodes aren't actual products - they're fees or
# adjustment entries: POST/DOT/C2 = postage & carriage charges,
# BANK CHARGES = bank fee, AMAZONFEE = marketplace fee,
# e.g. PADS was on an earlier version of this list but turned out
# to be a real product ("PADS TO MATCH ALL CUSHIONS"), so it's
# not included here.
def remove_non_products(df):
    junk_codes = ["POST", "DOT", "M", "C2", "BANK CHARGES",
                  "TEST001", "TEST002", "AMAZONFEE"]
    before = len(df)
    df = df[~df["stockcode"].str.upper().isin(junk_codes)]
    print(f"Removed non-product rows: {before - len(df)}")
    return df


# 6. Handle missing values

def handle_missing_values(df):
   
    before = len(df)
    df = df.dropna(subset=["description"])
    print(f"Dropped rows with missing description: {before - len(df)}")

    # missing customer_id just means a guest checkout, not bad
    # data Keep the row but flag it so
    # customer-level tasks (like RFM/segmentation) can filter it
    # out later if needed
    df["has_customer_id"] = df["customer_id"].notna()
    print(f"Rows without customer id (kept, flagged): {(~df['has_customer_id']).sum()}")
    return df


# 7. Remove invalid quantity/price values

def remove_invalid_values(df):
    before = len(df)
    df = df[df["quantity"] != 0]          # 0 items bought doesn't make sense
    df = df[df["price"] > 0]              # 0 or negative price doesn't make sense
    # negative quantity is only valid if the order is a cancellation (a return)
    df = df[~((df["quantity"] < 0) & (~df["is_cancelled"]))]
    print(f"Removed invalid qty/price rows: {before - len(df)}")
    return df



# 8. Handle outliers

def remove_extreme_outliers(df):
    before = len(df)
    extra_junk_codes = ["ADJUST", "D", "B", "CRUK"]
    df = df[~df["stockcode"].str.upper().isin(extra_junk_codes)]
    # implausible single-unit price with no customer attached
    df = df[~((df["quantity"] == 1) & (df["price"] > 500) & (df["customer_id"].isna()))]
    print(f"Removed extreme outlier rows: {before - len(df)}")
    return df



# Run the pipeline

df = load_data("/home/dell/ML-Learning/datasets/online_retail_II.csv")
print("Raw shape:", df.shape)

df = fix_types(df)
df = remove_duplicates(df)
df = flag_cancellations(df)
df = remove_non_products(df)
df = handle_missing_values(df)
df = remove_invalid_values(df)

# check the actual distribution before deciding on outlier cutoffs
print("\nQuantity stats before outlier removal:")
print(df["quantity"].describe())
print("\nPrice stats before outlier removal:")
print(df["price"].describe())

df = remove_extreme_outliers(df)


print("\nFinal shape:", df.shape)
print("\nRemaining missing values:\n", df.isnull().sum())
print("\nRemaining duplicates:", df.duplicated().sum())

# Save the cleaned dataset to a new CSV file
df.to_csv("/home/dell/ML-Learning/datasets/online_retail_II_cleaned.csv", index=False)
print("\nSaved cleaned file: online_retail_II_cleaned.csv")

Raw shape: (1067371, 8)
Removed duplicates: 34335
Cancelled orders flagged: 19104
Removed non-product rows: 5333
Dropped rows with missing description: 4262
Rows without customer id (kept, flagged): 228947
Removed invalid qty/price rows: 1730

Quantity stats before outlier removal:
count    1.021711e+06
mean     1.048969e+01
std      1.690453e+02
min     -8.099500e+04
25%      1.000000e+00
50%      3.000000e+00
75%      1.000000e+01
max      8.099500e+04
Name: quantity, dtype: float64

Price stats before outlier removal:
count    1.021711e+06
mean     3.410502e+00
std      1.360344e+01
min      1.000000e-03
25%      1.250000e+00
50%      2.100000e+00
75%      4.130000e+00
max      1.106206e+04
Name: price, dtype: float64
Removed extreme outlier rows: 259

Final shape: (1021452, 10)

Remaining missing values:
 invoice                 0
stockcode               0
description             0
quantity                0
invoicedate             0
price                   0
customer_id        2272